In [1]:
bpw = [4,3.923,4.27,3.936,3.862,3.514,4.171,4.282,4.173,4.055]
bpc = [0.495,0.51,0.48,0.479,0.489,0.46,0.482,0.47,0.484,0.466]
sum(bpw)/len(bpw), sum(bpc)/len(bpc)

(4.0186, 0.48149999999999993)

In [2]:
import ollama
import re
import time
import math
from collections import Counter

MODEL_NAME = "llama3"

group_0 = set("ISHLCFWPVXJishlcfwpvxj")
group_1 = set("NRDUMYGBKQZnrdumygbkqz")

bad_words = {"the","of","in","and","are","is","be","to","for","on","with"}

# ================================
# ENCODE / DECODE
# ================================
def encode(secret):
    table = {
        ' ': '1010','e':'1100','t':'1011','a':'1001','o':'0111',
        'i':'0110','n':'0100','s':'0011','r':'0010','h':'0001',
        'd':'11111','l':'11110','u':'11100','c':'11011','m':'11010',
        'f':'10000','y':'01011','w':'01010','g':'00001',
        'p':'111011','b':'111010','v':'100010','k':'000001',
        'x':'000000','q':'1000110','j':'10001111','z':'10001110'
    }
    return ''.join(table.get(c.lower(), '') for c in secret)

def decode(bits):
    mapping = {
        '10001111':'j','10001110':'z','1000110':'q',
        '111011':'p','111010':'b','100010':'v','000001':'k','000000':'x',
        '11111':'d','11110':'l','11100':'u','11011':'c','11010':'m',
        '10000':'f','01011':'y','01010':'w','00001':'g',
        '1010':' ','1100':'e','1011':'t','1001':'a',
        '0111':'o','0110':'i','0100':'n','0011':'s',
        '0010':'r','0001':'h'
    }

    keys = sorted(mapping.keys(), key=len, reverse=True)

    i, out = 0, ""
    while i < len(bits):
        for k in keys:
            if bits[i:i+len(k)] == k:
                out += mapping[k]
                i += len(k)
                break
        else:
            i += 1
    return out

# ================================
# WORD → BITS
# ================================
def word_to_bits(word):
    bits = ""
    for ch in word:
        if ch in group_0:
            bits += '0'
        elif ch in group_1:
            bits += '1'
    return bits

# ================================
# GENERATE TEXT
# ================================
def generate_text(seed):
    prompt = f"""
Continue the text with a natural paragraph.

Text: {seed}

Use meaningful, varied vocabulary. Avoid repetition.
"""
    res = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.85, "num_gpu": 1}
    )
    return res['message']['content']

# ================================
# EXTRACT WORDS
# ================================
def extract_words(text):
    return re.findall(r'\b[a-zA-Z]+\b', text)

# ================================
# CONTEXT SCORE
# ================================
def context_score(word, context_words):
    if not context_words:
        return 1
    return 1 if word.lower() not in context_words else 0.5

# ================================
# SCORE WORD
# ================================
def score_word(word, bits, context_words):

    wbits = word_to_bits(word)

    if len(wbits) == 0:
        return 0, 0

    if not bits.startswith(wbits):
        return 0, 0

    density = len(wbits) / len(word)
    bit_score = (len(wbits) ** 2) * density

    # penalize bad words
    if word.lower() in bad_words:
        bit_score *= 0.4

    # context awareness
    ctx = context_score(word, context_words)

    # repetition penalty
    freq = context_words.count(word.lower())
    repetition_penalty = 1 / (1 + freq)

    final_score = bit_score * ctx * repetition_penalty

    return final_score, len(wbits)

# ================================
# STEGO GENERATION
# ================================
def generate_stego(seed, bits):

    stego = seed
    payload_words = []
    remaining = bits

    context_words = []

    while remaining:

        text = generate_text(stego)
        words = extract_words(text)

        best_word = None
        best_score = 0
        best_len = 0

        for w in words:

            score, length = score_word(w, remaining, context_words)

            if score > best_score:
                best_word = w
                best_score = score
                best_len = length

        if best_word is None:
            continue

        stego += " " + best_word
        payload_words.append(best_word)
        context_words.append(best_word.lower())

        remaining = remaining[best_len:]

        print("Remaining bits:", len(remaining))

    return stego, payload_words

# ================================
# EXTRACTION
# ================================
def extract_bits(words):
    bits = ""
    for w in words:
        bits += word_to_bits(w)
    return bits

# ================================
# METRICS
# ================================
def compute_metrics(stego, bits, start):
    words = len(stego.split())
    chars = len(stego.replace(" ", ""))

    BPW = round(len(bits)/words, 3)
    BPC = round(len(bits)/chars, 3)
    log_ppl = round(math.log(words + 1), 3)
    t = round(time.time() - start, 2)

    return BPW, BPC, log_ppl, t

# ================================
# MAIN
# ================================
if __name__ == "__main__":

    secret = input("Enter secret:\n")
    seed = input("Enter seed:\n")

    start = time.time()

    bits = encode(secret)
    print(bits)
    print("Total bits:", len(bits))

    stego, payload_words = generate_stego(seed, bits)

    print("\n--- STEGO TEXT ---\n")
    print(stego)

    extracted_bits = extract_bits(payload_words)
    decoded = decode(extracted_bits)

    print("\n🔓 Extracted Message:\n", decoded)

    BPW, BPC, log_ppl, t = compute_metrics(stego, bits, start)

    print("\n--- Metrics ---")
    print("BPW:", BPW)
    print("BPC:", BPC)
    print("Log PPL:", log_ppl)
    
    print("Time:", t, "sec")

Enter secret:
 manoj
Enter seed:
 people in nowadays are so busy 


1101010010100011110001111
Total bits: 25
Remaining bits: 21
Remaining bits: 17
Remaining bits: 11
Remaining bits: 7
Remaining bits: 2
Remaining bits: 0

--- STEGO TEXT ---

people in nowadays are so busy  busy stress relentless modern feeling room

🔓 Extracted Message:
 manoj

--- Metrics ---
BPW: 2.083
BPC: 0.403
Log PPL: 2.565
Time: 135.91 sec


In [ ]:
bpw = [2.5,2.17,2.13,2.2,2.18,1.85,2.47,2.08,1.86,1.77,
      2.73,2.73,2.6,2.66,2.83,2.75,2.91,3.12,2.91,2.62,
      2.92,3.38,2.96,3.59,3.55,3.43,3.16,3.29,3.07,3,
      3.05,3.01,2.92,2.96,3.37,3.91,
      3.55,
      3.5,
      3.51,
      3.53,
      3.7,3.9,3.61,3.54,
      3.65,3.645,3.89,3.72]  
bpc = [0.44,0.32,0.33,0.35,0.31,0.36,0.49,0.429,0.32,0.34,
      0.502,0.53,0.39,0.44,0.46,0.46,0.40,0.495,0.51,0.51,
      0.518,0.55,0.52,0.51,0.56,0.55,0.55,0.56,0.47,0.49,
      0.51,0.51,0.52,0.53,0.48,0.56,
      0.53,
      0.54,
      0.57,
      0.57,
      0.63,0.61,0.58,0.6,
      0.6,0.59,0.61,0.59]   
log_ppl=[3.04,3.17,3.13,3.25,3.13,3.33,3.091,3.17,3.40,3.46,
        3.63,3.66,3.71,3.69,3.58,3.611,3.6,3.49,3.61,3.68,
        3.93,3.80,3.95,3.76,3.78,3.85,3.91,3.89,3.97,3.97,
        4.11,4.12,4.15,4.11,3.99,3.89,
        4.15,
        4.37,
        4.47,
        4.65,
        4.71,4.65,4.75,4.74,
        4.82,4.82,4.76,4.78] 
time = [62.79,71.84,65.64,109.05,120.48,124.84,95.4,88.48,135.25,101.3,
       177.42,173.93,231.83,188.71,152.71,144.31,183.96,125.21,132.38,137.9,
       195.5,193.09,225.55,173.48,171.14,207.21,271.28,243.66,438,271.98,
       297.45,351.56,357.52,382.16,353.07,315.2,
       358.79,
       438.29,
       471.52,
       560.39,
       641.26,997.93,1258.92,789.55,
       1296.24,960.41,689.07,779.7]
print("bpw = ",sum(bpw)/len(bpw))
print("BPC = ",sum(bpc)/len(bpc))
print("log ppl = ", sum(log_ppl)/len(log_ppl))
print("time = ",sum(time)/len(time))

In [133]:
#mistral
bpw = [2.2,2,2.19,2.3,2.22] #,3.24]
bpc = [0.39,0.38,0.37,0.43,0.39] #,0.53]
log_ppl=[3.17,3.22,3.1,3.21,3.13] #,4.07]
time = [58.62,42.73,41.01,61.84,45.17] #,242.21]
print("log ppl = ", sum(log_ppl)/len(log_ppl))
print("time = ",sum(time)/len(time))

log ppl =  3.3166666666666664
time =  81.92999999999999
